# Waxal ASR Kaggle Baseline Runner

Purpose: run the zero-shot Whisper smoke test and full baseline on Kaggle's free GPU notebook environment.

## 1. Runtime Check

In Kaggle, enable GPU from notebook settings before running inference.

In [ ]:
import torch, platform
print('Python:', platform.python_version())
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Clone Repo

Clone the published GitHub repository into the Kaggle runtime.

In [ ]:
REPO_URL = 'https://github.com/Maziko-M98/waxal-asr-challenge.git'
PROJECT_DIR = '/kaggle/working/waxal_asr_challenge'

!rm -rf {PROJECT_DIR}
!git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}

## 3. Install Dependencies

In [ ]:
!python -m pip install --upgrade pip -q
!pip install -e . -q
!pip install -r requirements.txt -q

## 4. Attach Zindi CSVs

Recommended Kaggle setup: create a private Kaggle Dataset containing `Train.csv`, `Test.csv`, and `SampleSubmission.csv`, then attach it to this notebook. Set `ZINDI_INPUT_DIR` to that dataset path.

In [ ]:
from pathlib import Path
import shutil

ZINDI_INPUT_DIR = Path('/kaggle/input/YOUR_PRIVATE_ZINDI_DATASET')
data_dir = Path('data/zindi')
data_dir.mkdir(parents=True, exist_ok=True)

if not ZINDI_INPUT_DIR.exists():
    raise FileNotFoundError('Set ZINDI_INPUT_DIR to your attached private Kaggle Dataset path.')

for name in ['Train.csv', 'Test.csv', 'SampleSubmission.csv']:
    shutil.copy2(ZINDI_INPUT_DIR / name, data_dir / name)
    print('copied', name)

## 5. EDA And Dry Run

In [ ]:
!python -m waxal_asr.inspect_zindi_csv --data-dir data/zindi
!python -m waxal_asr.eda_report --data-dir data/zindi --out-dir reports/eda
!python -m waxal_asr.infer_whisper --config configs/baselines/zero_shot_whisper_large_v3_turbo.json --dry-run

## 6. Smoke Inference

In [ ]:
!python -m waxal_asr.infer_whisper \
  --config configs/baselines/zero_shot_whisper_large_v3_turbo.json \
  --max-samples 20 \
  --output submissions/smoke_20_zero_shot_whisper.csv \
  --raw-predictions submissions/raw/smoke_20_zero_shot_whisper_raw.csv

!python - <<'PY'
import pandas as pd
df = pd.read_csv('submissions/smoke_20_zero_shot_whisper.csv')
print(df.shape)
print(df.head())
PY

## 7. Full Baseline

In [ ]:
# !python -m waxal_asr.infer_whisper \
#   --config configs/baselines/zero_shot_whisper_large_v3_turbo.json \
#   --raw-predictions submissions/raw/baseline_001_zero_shot_whisper_large_v3_turbo_raw.csv